# Работа с файлами в Python

## Файлы

Файл — последовательность данных на диске. Путь указывает, где он находится.

- **Абсолютный путь** начинается от корня файловой системы (в Linux - `/`).
- **Относительный путь** вычисляется от текущей рабочей директории (в Linux - `.`).

Кодировка определяет, как символы переводятся в байты. Для текстовых файлов по умолчанию используется `utf-8`.

Создадим файл в текущей директории:

In [1]:
%%writefile verse.txt
Дай, Джим, на счастье, лапу мне,
Такую лапу не видал я сроду.
Давай с тобой полаем при луне
На тихую, бесшумную погоду.
Дай, Джим, на счастье, лапу мне.

Writing verse.txt


И посмотрим на содержимое директории:

In [30]:
!ls -lah .

total 32K
drwxr-xr-x 1 root root 4.0K Oct 16 18:06 .
drwxr-xr-x 1 root root 4.0K Oct 16 17:15 ..
drwxr-xr-x 3 root root 4.0K Oct 16 18:06 abra
drwxr-xr-x 4 root root 4.0K Oct  2 13:50 .config
-rw-r--r-- 1 root root   40 Oct 16 17:52 created_once.txt
drwxr-xr-x 1 root root 4.0K Oct  2 13:50 sample_data
-rw-r--r-- 1 root root   78 Oct 16 17:51 verse.txt
-rw-r--r-- 1 root root   78 Oct 16 17:57 verse_uppercase.txt


А также получим абсолютный путь до этого файла:

In [3]:
!realpath verse.txt

/content/verse.txt


## `open()`

Это основная функция для работы с файлами в Python.

Общая форма:

```python
open(file, mode="r", encoding=None)
```
Откроем файл со стихом:

In [4]:
f = open("verse.txt")  # То же самое, что и open("verse.txt", "r", "utf-8")
print(f)
print(type(f))

<_io.TextIOWrapper name='verse.txt' mode='r' encoding='utf-8'>
<class '_io.TextIOWrapper'>


`mode` отвечает за режим открытия файлов.

У текстовых файлов следующие основные режимы:

| Режим | Назначение | Если файла нет | Если файл есть |
|---|---|---|---|
| `"r"` | чтение | `FileNotFoundError` | читает |
| `"w"` | запись | создаёт | очищает перед записью |
| `"a"` | добавление в конец | создаёт | сохраняет содержимое |
| `"x"` | только создание | создаёт | `FileExistsError` |

Метод  `.readline()` читает одну строку из файла:


In [5]:
print(f.readline())
print(f.readline())

Дай, Джим, на счастье, лапу мне,

Такую лапу не видал я сроду.



Читать построчно полезно в случае больших файлов - мы экономим оперативную память: загрузил - обрабоотал - следующая строка.

Метод  `.read()` загружает всё содержимое файла:

In [6]:
f.read()

'Давай с тобой полаем при луне\nНа тихую, бесшумную погоду.\nДай, Джим, на счастье, лапу мне.\n'

Почему мы получили не полностью все содержимое? Файловый объект хранит позицию, до какого момента он отдал содержимое.

In [12]:
f.read()  # файл прочитан до конца - нечего возвращать

''


Вернуться в начало файла помогает метод `.seek()`:

In [13]:
f.seek(0)  # 0 - начало файла, можно указать и другое место

0

И теперь мы можем загрузить весь файл:

In [14]:
f.read()

'Дай, Джим, на счастье, лапу мне,\nТакую лапу не видал я сроду.\nДавай с тобой полаем при луне\nНа тихую, бесшумную погоду.\nДай, Джим, на счастье, лапу мне.\n'

Мы можем построчно обрабатывать файл в цикле `for`:

In [15]:
f.seek(0)

for line in f:
    print(line)

Дай, Джим, на счастье, лапу мне,

Такую лапу не видал я сроду.

Давай с тобой полаем при луне

На тихую, бесшумную погоду.

Дай, Джим, на счастье, лапу мне.



Файл обязательно надо закрыть - для этого используем `.close()`:

In [16]:
f.close()

In [17]:
f.closed

True

## Запись текста

Для записи файл надо открыть в режиме для записи:


In [18]:
f = open("verse.txt", "w")

Метод `.write()` возвращает число записанных символов. Повторное открытие в режиме `w` **перезапишет** файл.

In [19]:
written = f.write("Первая строка\nВторая строка\n")  # Внимание на \n - запись строк!
f.close()

print("Записано символов:", written)
print("Файл закрыт:", f.closed)  # Атрибут .closed показывает, закрыт ли файл

f = open("verse.txt")
content = f.read()
f.close()
print(repr(content))


Записано символов: 28
Файл закрыт: True
'Первая строка\nВторая строка\n'


Для добавления строк в существующий текст - режим `a`.

Запись идёт в конец, старые данные остаются:

In [20]:
f = open("verse.txt", "a")
f.write("Третья строка\n")
f.close()

f = open("verse.txt")
content = f.read()
f.close()
print(repr(content))

'Первая строка\nВторая строка\nТретья строка\n'


Создание без перезаписи - режим `x`. Этот режим полезен, когда существующий файл нельзя случайно уничтожить:

In [21]:
try:
    f = open("created_once.txt", "x")
    f.write("Файл создан безопасно")
except FileExistsError:
    print("Файл уже существует — не перезаписываем его")
finally:
    f.close()

f = open("created_once.txt", "x")  # Ошибка!

FileExistsError: [Errno 17] File exists: 'created_once.txt'

## Контекстные менеджеры


Контекстный менеджер `with` закрывает файл при выходе из блока — даже если внутри произошло исключение.

```python
with open(path) as file:  # file = open(path)
    data = file.read()
# здесь файл уже закрыт
```

Это основной и рекомендуемый способ работы с файлами.

In [23]:
with open("verse.txt", "r", encoding="utf-8") as file:
    content = file.read(14)  # Читаем 14 байт
    print("Внутри блока закрыт?", file.closed)

print("После блока закрыт?", file.closed)
print("Фрагмент:", repr(content))

Внутри блока закрыт? False
После блока закрыт? True
Фрагмент: 'Первая строка\n'


Это удобнее, чем `try...finally...`:

In [ ]:
try:
    file = open("verse.txt")
finally:
    file.close()

Один `with` может управлять несколькими файлами. Например, скопируем текст, одновременно читая источник и записывая результат.

In [24]:

with (
    open("verse.txt") as source,
    open("verse_uppercase.txt", "w") as target,
):
    for line in source:
        target.write(line.upper())

Проверим содержимое:

In [25]:
# %load verse_uppercase.txt
ПЕРВАЯ СТРОКА
ВТОРАЯ СТРОКА
ТРЕТЬЯ СТРОКА


## pathlib

Это библиотека Python, которая позволяет удобно работать с путями файлов и директорий в операционной системе.

`pathlib.Path` работает одинаково на Windows, macOS и Linux. Оператор `/` соединяет части пути без ручной подстановки слешей.

In [26]:
import pathlib  # from pathlib import Path

current_dir = pathlib.Path(".")  # текущая директория
print(current_dir.is_dir())  # True

verses = current_dir / "verse.txt"

print("Имя:", verses.name)
print("Основа имени:", verses.stem)
print("Расширение:", verses.suffix)
print("Родитель:", verses.parent)
print("Полный путь к файлу:", verses.absolute())

not_exists = current_dir / "some_non_existing_file"
print(not_exists.exists())

True
Имя: verse.txt
Основа имени: verse
Расширение: .txt
Родитель: .
Полный путь к файлу: /content/verse.txt
False


Метод `.mkdir(parents=True, exist_ok=True)` создаёт всю цепочку папок и не падает, если она уже существует.

In [27]:
new_dir = current_dir / "abra" / "cadabra"
print(new_dir.exists())  # False
print(new_dir.is_dir())  # False

new_dir.mkdir(parents=True)  # Создает все нужные директории
print(new_dir.exists())  # True
print(new_dir.is_dir())  # True
print(new_dir.absolute())

new_dir.mkdir(exist_ok=True)  # Без ошибки - путь уже существует
new_dir.mkdir()  # Ошибка

False
False
True
True
/content/abra/cadabra


FileExistsError: [Errno 17] File exists: 'abra/cadabra'

Методы `.write_text()` и `.read_text()` удобны для небольших файлов, когда не так много данных для записи/чтения:

In [28]:
summary_path = new_dir / "summary.txt"  # /content/abra/cadabra/summary.txt
summary_path.write_text("Выручка: 125000\nВалюта: BYN\n")

print(summary_path.read_text(encoding="utf-8"))
print("Существует:", summary_path.exists())
print("Это файл:", summary_path.is_file())

Выручка: 125000
Валюта: BYN

Существует: True
Это файл: True


Метод `.iterdir()` перебирает непосредственное содержимое каталога:


In [29]:
for item in current_dir.iterdir():  # /content
    print(item.name)
    print(item.absolute())
    print(item.is_dir())
    print(item.is_file())

.config
/content/.config
True
False
verse_uppercase.txt
/content/verse_uppercase.txt
False
True
abra
/content/abra
True
False
created_once.txt
/content/created_once.txt
False
True
verse.txt
/content/verse.txt
False
True
sample_data
/content/sample_data
True
False


Метод `.glob()` ищет по шаблону в текущем каталоге, например `"*.txt"` - все файлы с расширением `.txt`:

In [35]:
for file in current_dir.glob("*.txt"):
    print(file.name)
    print(file.absolute())

verse_uppercase.txt
/content/verse_uppercase.txt
created_once.txt
/content/created_once.txt
verse.txt
/content/verse.txt


In [33]:
%%writefile hello.py
print("Hello, world!")

Writing hello.py


In [37]:
for file in current_dir.glob("*.py"):
    print(file.name)
    print(file.absolute())

.config
/content/.config


Метод `.rglob()` ищет рекурсивно во всех вложенных каталогах:

In [38]:
for file in current_dir.rglob("*.txt"):
    print(file.name)
    print(file.absolute())

verse_uppercase.txt
/content/verse_uppercase.txt
created_once.txt
/content/created_once.txt
verse.txt
/content/verse.txt
summary.txt
/content/abra/cadabra/summary.txt


Для переименования файла используем `.rename()`:


In [39]:
new_verses_path = current_dir / "new_verses.txt"
print(new_verses_path.exists())  # False
verses.rename(new_verses_path)
print(new_verses_path.exists())  # True
print(verses.exists())  # False

False
True
False


Метод `.unlink()` удаляет файл:


In [40]:
new_verses_path.unlink()
print(new_verses_path.exists())  # False

False


А метод `.rmdir()` удаляет **пустой** каталог:

In [41]:
try:
    new_dir.rmdir()  # /content/abra/cadabra
except Exception as e:
    print(e)
    print(type(e))  # OSError


summary_path.unlink()
new_dir.rmdir()  # Теперь ОК

[Errno 39] Directory not empty: 'abra/cadabra'


## JSON

*JSON* - формат передачи данных, особенно популярный при обмене сообщениями между серверами и клиентскими приложениями.

JSON хранит вложенные структуры из объектов, массивов, строк, чисел, логических значений и `null`.

Соответствия с Python:

| JSON | Python |
|---|---|
| object | `dict` |
| array | `list` |
| string | `str` |
| number | `int` / `float` |
| true / false | `True` / `False` |
| null | `None` |

Кортеж после сохранения и загрузки станет списком. Остальные типы требуют явного преобразования.

In [42]:
import json

course = {
    "title": "Python",
    "year": 2026,
    "topics": ["files", "json", "csv"],
    "active": True,
    "teacher": None,
}

json_text = json.dumps(course, ensure_ascii=False, indent=2)
print(json_text)

{
  "title": "Python",
  "year": 2026,
  "topics": [
    "files",
    "json",
    "csv"
  ],
  "active": true,
  "teacher": null
}


`.dumps()` и `.loads()` работают со **строками**, а `.dump()` и `.load()` — с **файловыми объектами**.

In [43]:
restored_from_text = json.loads(json_text)
print(restored_from_text["topics"])
print(type(restored_from_text))

['files', 'json', 'csv']
<class 'dict'>


In [44]:
json_path = current_dir / "course.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(course, f, ensure_ascii=False, indent=2)

with open(json_path, encoding="utf-8") as f:
    loaded_course = json.load(f)

print(loaded_course)
assert loaded_course == course

{'title': 'Python', 'year': 2026, 'topics': ['files', 'json', 'csv'], 'active': True, 'teacher': None}


Некорректный JSON вызывает `json.JSONDecodeError`. Ошибку лучше перехватывать рядом с чтением и сообщать, какой файл не удалось разобрать.

In [45]:
broken_json_path = current_dir / "broken.json"
broken_json_path.write_text('{"name": "Alice",}', encoding="utf-8")  # Лишняя запятая

try:
    with open(broken_json_path, encoding="utf-8") as file:
        json.load(file)
except json.JSONDecodeError as error:
    print(f"Некорректный JSON в строке {error.lineno}, столбце {error.colno}")  # У ошибки JSONDecodeError есть свои атрибуты

Некорректный JSON в строке 1, столбце 17


## CSV


*CSV* хранит таблицу как строки с разделителями. Часто встречаются запятая (отсюда и CSV - comma separated values), точка с запятой и табуляция. Первая строка может хранить заголовки таблицы - имена колонок.


In [46]:
%%writefile students.csv
name,score,city
Анна,92,Минск
Иван,85,Брест
Мария,97,Гродно

Writing students.csv


В Python есть модуль `csv`, упрощающий работу с такими таблицами:


In [47]:
import csv

csv_path = current_dir / "students.csv"

with open(csv_path, newline="") as f:
    for row in csv.reader(f, delimiter=","):
        print(row)

['name', 'score', 'city']
['Анна', '92', 'Минск']
['Иван', '85', 'Брест']
['Мария', '97', 'Гродно']


При открытии CSV рекомендуется указывать `newline=""`: тогда модуль `csv` сам корректно управляет переводами строк на разных ОС.

Важно: значения из CSV читаются как строки, поэтому числа нужно преобразовывать явно.

`DictReader` превращает каждую строку в словарь с именами колонок:

In [48]:
with open(csv_path, newline="") as file:
    reader = csv.DictReader(file)
    for row in reader:
        print(row)

{'name': 'Анна', 'score': '92', 'city': 'Минск'}
{'name': 'Иван', 'score': '85', 'city': 'Брест'}
{'name': 'Мария', 'score': '97', 'city': 'Гродно'}


Запишем новый `.csv` файл:

In [49]:
products_path = current_dir / "products.csv"
rows = [
    ["product", "price", "comment"],  # Заголовки
    ["Книга", 25.5, "Бумага, твёрдый переплёт"],
    ["Ручка", 2.4, "Синяя"],
]

with open(products_path, "w", newline="") as file:  # Режим записи
    writer = csv.writer(file, delimiter=";")  # Разделитель ;
    writer.writerows(rows)  # Запишем все строки сразу

Или с помощью `DictWriter`:

In [51]:
students = [
    {"name": "Анна", "score": 92, "city": "Минск"},
    {"name": "Иван", "score": 85, "city": "Брест"},
    {"name": "Мария", "score": 97, "city": "Гродно"},
]

new_csv_path = current_dir / "students_new.csv"
fieldnames = ["name", "score", "city"]  # Важен порядок!

with open(new_csv_path, "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames, delimiter=";")
    writer.writeheader()
    writer.writerows(students)

print(new_csv_path.read_text(encoding="utf-8"))

name;score;city
Анна;92;Минск
Иван;85;Брест
Мария;97;Гродно



## Практика


Есть список покупок. Выполните следующие действия:

1. Создайте каталог `shop` внутри `demo_dir`.
2. Сохраните список в `purchases.json`.
3. Прочитайте JSON и вычислите поле `total = quantity * price` для каждой покупки.
4. Сохраните таблицу `report.csv` со столбцами `product`, `quantity`, `price`, `total`.
5. Выведите содержимое CSV.

Используйте `Path`, `with`, `json` и `csv.DictWriter`.

In [ ]:
purchases = [
    {"product": "Кофе", "quantity": 2, "price": 18.50},
    {"product": "Чай", "quantity": 3, "price": 9.20},
    {"product": "Шоколад", "quantity": 4, "price": 4.80},
]

# Попробуйте решить задачу здесь.
# shop_dir = ...
# json_report_path = ...
# csv_report_path = ...